# Data Analytics Case Study 3: Project
Bruno Sassi - NF1008627

Maria Aguilar - NF1009877

Luciana Popa - NF1005554

Tiago Ceolato – NF1006634

In [1]:
# Importing libraries for the GOOGL stock price prediction, using
# Multiple linear regression and Gradient boosting.

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.metrics import root_mean_squared_error

In [2]:
# Loading and preparing the dataset:

df = pd.read_csv("alphabet_prepared.csv")
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date').reset_index(drop=True)

FileNotFoundError: [Errno 2] No such file or directory: 'alphabet_prepared.csv'

#EDA

In [ ]:
df.info()

In [ ]:
df.head()

In [ ]:
df.isnull().sum()

In [ ]:
df.describe().T

In [ ]:
plt.figure(figsize=(12,6))
plt.plot(df['Date'], df['Price'], label='Closing Price')
plt.title("GOOGL Stock Price Over Time")
plt.xlabel("Date")
plt.ylabel("Price")
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(10,5))
df['Return'].hist(bins=50)
plt.title("Distribution of Daily Returns")
plt.xlabel("Daily Return")
plt.show()

plt.figure(figsize=(12,6))
plt.plot(df['Date'], df['Return'], label='Daily Return')
plt.title("Daily Returns Over Time")
plt.show()


In [ ]:
plt.figure(figsize=(12,6))
plt.plot(df['Date'], df['Price'], label='Price', alpha=0.8)
plt.plot(df['Date'], df['MA20'], label='MA20')
plt.plot(df['Date'], df['MA50'], label='MA50')
plt.plot(df['Date'], df['MA200'], label='MA200')
plt.title("Price with Moving Averages")
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(12,8))
corr = df[['Price','Open','High','Low','Vol.','Return','MA10','MA20','MA50','MA200','RSI','MACD','Signal']].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation Heatmap")
plt.show()


In [ ]:
fig, ax1 = plt.subplots(figsize=(12,6))

ax1.plot(df['Date'], df['Price'], color='blue', label='Price')
ax1.set_ylabel("Price", color="blue")
ax2 = ax1.twinx()
ax2.bar(df['Date'], df['Vol.'], alpha=0.3, color='gray', label='Volume')
ax2.set_ylabel("Volume", color="gray")

plt.title("Price and Trading Volume")
plt.show()

In [ ]:
plt.figure(figsize=(12,4))
plt.plot(df['Date'], df['RSI'], label='RSI')
plt.axhline(70, color='red', linestyle='--')
plt.axhline(30, color='green', linestyle='--')
plt.title("RSI Over Time")
plt.legend()
plt.show()

plt.figure(figsize=(12,4))
plt.plot(df['Date'], df['MACD'], label='MACD')
plt.plot(df['Date'], df['Signal'], label='Signal')
plt.axhline(0, color='black', linestyle='--')
plt.title("MACD and Signal Line")
plt.legend()
plt.show()


#Testing

In [ ]:
# Creating the target variable (which is the next-day closing price)
df['y_next'] = df['Price'].shift(-1)

In [ ]:
# Selecting features for H1, including lagged prices only:

features_A = ['Price_lag1', 'Price_lag5']

# Selecting features for H2 and H3, which are the lagged proces and the technical indicators:

features_B = features_A + [
    'MA10','MA20','MA50','MA200',
    'RSI','MACD','Signal','Volatility20','Vol.'
]

# The MovingAvg_10day is the short-term average, the MovingAvg_20day is the medium-term average.
# MovingAvg_50day is a medium/long-term average, and MovingAvg_200day is a long-term average.


In [ ]:
# Dropping rows with missing/null values:

df = df.dropna(subset=features_B + ['y_next'])

X_A = df[features_A]   # for model A
X_B = df[features_B]   # for model B and gradient boosting
y = df['y_next']     # target (next day closing price)
dates = df['Date']

In [ ]:
# Separating data chronologically:

train_idx = dates < "2024-01-01"   # our training data (2021-2023)
valid_idx = (dates >= "2024-01-01") & (dates < "2025-01-01") # just a validation data (2024)
test_idx  = dates >= "2025-01-01"   # our testing data (2025)

In [ ]:
# Calculating evaluation metrics:

def evaluate_model(name, y_true, y_pred):
    return {
        "Model": name,
        "RMSE": root_mean_squared_error(y_true, y_pred),  # this will be our avg predictor error
        "MAE": mean_absolute_error(y_true, y_pred),    # error in monetary value
        "R2": r2_score(y_true, y_pred)    # % of the variance explained
    }

results = []

**Multiple Linear Regression**

In [ ]:
# Model A (H1):

pipe_lr_A = Pipeline([
    ("scaler", StandardScaler()),      # normalizing data so we can compare them properly
    ("lr", LinearRegression())
])
pipe_lr_A.fit(X_A[train_idx], y[train_idx])
results.append(evaluate_model("LR_A (lags) - Valid", y[valid_idx], pipe_lr_A.predict(X_A[valid_idx])))
results.append(evaluate_model("LR_A (lags) - Test",  y[test_idx],  pipe_lr_A.predict(X_A[test_idx])))

In [ ]:
# Model B (H2):

pipe_lr_B = Pipeline([
    ("scaler", StandardScaler()),
    ("lr", LinearRegression())
])
pipe_lr_B.fit(X_B[train_idx], y[train_idx])
results.append(evaluate_model("LR_B (lags+tech) - Valid", y[valid_idx], pipe_lr_B.predict(X_B[valid_idx])))
results.append(evaluate_model("LR_B (lags+tech) - Test",  y[test_idx],  pipe_lr_B.predict(X_B[test_idx])))

In [ ]:
# Most relevant coefficients obtained from Model B:

coef_B = pd.Series(pipe_lr_B.named_steps['lr'].coef_, index=features_B).sort_values(key=abs, ascending=False)
print("\nTop 10 Linear Regression Coefficients (Model B):")
print(coef_B.head(10))

With the previous output we have the expected change
in the target variable (next day closing price)
when there is a increase of 1 unit on the predictor, while keeping other variables constant.
MA20 shows that for every 1 unit increase in the 20 day moving avg, the predicted target will increase by 27.97 units.
Signal shows that for every 1 unit increase in signal, the predictor will decrease by 13.99 units.
We can see that the most influential coefficients are MA20, Signal and also MACD, which are larger compared to the others.

**Gradient Boosting (machine learning)**

In [ ]:
# Model C (H3):

gb = GradientBoostingRegressor(random_state=42)

# Adjusting the parameters:

param_grid = {
    "n_estimators": [200, 400],    # number of trees
    "max_depth": [2, 3],
    "learning_rate": [0.05, 0.1]
}


grid = GridSearchCV(
    gb,
    param_grid,
    cv=[(train_idx, valid_idx)],  # train/valid split
    scoring="neg_root_mean_squared_error"
)
grid.fit(X_B, y)

# Best gradient boosting model:

best_gb = grid.best_estimator_
results.append(evaluate_model("GB - Valid", y[valid_idx], best_gb.predict(X_B[valid_idx])))
results.append(evaluate_model("GB - Test",  y[test_idx],  best_gb.predict(X_B[test_idx])))

In [ ]:
# Showing feature importances from the gradient boosting model:

gb_importance = pd.Series(best_gb.feature_importances_, index=features_B).sort_values(ascending=False)
print("\nTop 10 Gradient Boosting Feature Importances:")
print(gb_importance.head(10))

The previuos output shows how much each feature contributes to reducing error in the ensemble trees.
Price_lag1 is the main feature, with 85.85% importance.
 It indicates that the previous day price (Price_lag1) is the most predictive feature in this model.
When analysing data with this model we can get the dominant feature, even if the linear coefficient was smaller, for example.

**Testing Results**

In [ ]:
# Final results table:

results_df = pd.DataFrame(results)
print("\nModel Evaluation Results:")
print(results_df)

In general, there is a great proportion of variance being explained, as showed by the high R2s.
By adding technical indicators on LR_B the performance increased a bit, but decreased RMSE as well.
The opposite happened to the test performance. Can indicate overfitting.
The gradient boosting model performs better than the previous models, with less errors and higher R2s.
In the end, the gradient boosting test seems to be more robust, capturing the nonlinear patterns that the LR may miss.

In [ ]:
plt.figure(figsize=(8,5))
plt.bar(results_df["Model"], results_df["RMSE"], color="skyblue")
plt.xticks(rotation=45)
plt.title("RMSE Comparison by Model")
plt.ylabel("RMSE")
plt.show()

Gradient Boosting (GB) has the lowest RMSE in both validation and test

In [ ]:
plt.figure(figsize=(8,5))
plt.bar(results_df["Model"], results_df["R2"], color="lightgreen")
plt.xticks(rotation=45)
plt.title("R² Comparison by Model")
plt.ylabel("R²")
plt.show()

GB achieves R² ≈ 0.97+, higher than both linear regressions. This confirms GB generalizes best.

In [ ]:
y_test_true = y[test_idx]
y_test_pred_gb = best_gb.predict(X_B[test_idx])

plt.figure(figsize=(12,6))
plt.plot(dates[test_idx], y_test_true, label="Real", color="blue")
plt.plot(dates[test_idx], y_test_pred_gb, label="Predicted (GB)", color="red")
plt.title("Test Set Predictions - Gradient Boosting")
plt.xlabel("Date")
plt.ylabel("Closing Price")
plt.legend()
plt.show()

Lines overlap closely → GB tracks the overall trend (ups, downs, reversals) very well.

**Model Interpretetion**

In [ ]:
gb_importance = pd.Series(best_gb.feature_importances_, index=features_B).sort_values(ascending=True)

plt.figure(figsize=(8,6))
gb_importance.plot(kind="barh", color="purple")
plt.title("Feature Importance - Gradient Boosting")
plt.show()

Price_lag1 dominates → yesterday’s price is the strongest predictor.

In [ ]:
residuals = y_test_true - y_test_pred_gb

plt.figure(figsize=(10,5))
plt.scatter(y_test_pred_gb, residuals, alpha=0.6)
plt.axhline(0, color="red", linestyle="--")
plt.xlabel("Predicted Values")
plt.ylabel("Residuals")
plt.title("Residual Plot - Gradient Boosting")
plt.show()

Residuals are spread around 0 (red line) without strong patterns.
This suggests GB is not systematically biased (no consistent over/underestimation).
Some larger residuals exist (outliers), but mostly errors are small.

In [ ]:
plt.figure(figsize=(6,6))
plt.scatter(y_test_true, y_test_pred_gb, alpha=0.6)
plt.plot([y_test_true.min(), y_test_true.max()],
         [y_test_true.min(), y_test_true.max()],
         'k--', lw=2)
plt.xlabel("Actual Values")
plt.ylabel("Predicted Values")
plt.title("Actual vs Predicted - Gradient Boosting")
plt.show()

Points align almost perfectly along the 45° dashed line. Indicates very high predictive accuracy